# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/24f2001824/ml-flyrank/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

I selected numerical and categorical fields that are available before prediction. I excluded identifiers and fields that directly describe the target or future outcome. Numerical missing values are filled with the median and categorical missing values are filled with the most frequent category. Categorical features are one-hot encoded.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
!git clone https://github.com/24f2001824/ml-flyrank.git


from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

df = pd.read_csv("/content/ml-flyrank/data/raw/content_refresh_anonymized.csv")


df["is_declining_label"] = (
    df["trend_direction"].str.lower() == "down"
).astype(int)

features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "scroll_events_90d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "content_type",
    "main_intent"
]

X = df[features]
y = df["is_declining_label"]

numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

X_transformed = preprocessor.fit_transform(X)

print("Rows:", X.shape[0])
print("Original features:", X.shape[1])
print("Numerical features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Transformed shape:", X_transformed.shape)

Cloning into 'ml-flyrank'...
remote: Enumerating objects: 154, done.
remote: Counting objects: 100% (154/154), done.
remote: Compressing objects: 100% (110/110), done.
remote: Total 154 (delta 60), reused 96 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (154/154), 1.89 MiB | 9.56 MiB/s, done.
Resolving deltas: 100% (60/60), done.
Rows: 30000
Original features: 21
Numerical features: 19
Categorical features: 2
Transformed shape: (30000, 26)


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

The numerical features describe search demand, content size, visibility, engagement, position, freshness, and traffic. Missing numerical values are filled with the median, while missing categorical values are filled with the most frequent category.

The selected features are intended to be available before the prediction decision. Fields such as client_id and content_id are identifiers rather than useful predictive signals, so they are excluded.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_notes = pd.DataFrame({
    "feature": features,
    "type": [df[f].dtype for f in features],
    "missing_values": [df[f].isna().sum() for f in features],
    "available_before_prediction": ["Yes"] * len(features)
})

feature_notes

,feature,type,missing_values,available_before_prediction
0,search_volume,float64,2468,Yes
1,competition,float64,2468,Yes
2,cpc,float64,2468,Yes
3,word_count,float64,7699,Yes
4,char_count,float64,7699,Yes
5,impressions_90d,int64,0,Yes
6,clicks_90d,int64,0,Yes
7,pageviews_90d,int64,0,Yes
8,sessions_90d,int64,0,Yes
9,users_90d,int64,0,Yes


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

I checked the feature list for fields that directly reveal the target, identify a page or client, or contain information that should not be used as a predictive feature.

The label is derived from trend_direction, so trend_direction and trend_pct are excluded. The label itself, is_declining_label, is also excluded from the features. Client_id and content_id are identifiers and are excluded to avoid learning identifier-specific patterns.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
leakage_fields = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "client_id",
    "content_id"
]

leakage_check = pd.DataFrame({
    "field": leakage_fields,
    "in_features": [field in features for field in leakage_fields],
    "reason": [
        "Used to derive the target label",
        "Direct trend information related to the target",
        "Target label itself",
        "Identifier, not a content signal",
        "Identifier, not a content signal"
    ]
})

leakage_check

,field,in_features,reason
0,trend_direction,False,Used to derive the target label
1,trend_pct,False,Direct trend information related to the target
2,is_declining_label,False,Target label itself
3,client_id,False,"Identifier, not a content signal"
4,content_id,False,"Identifier, not a content signal"


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

- `trend_direction` — excluded because it is used to create the target label.
- `trend_pct` — excluded because it directly describes the trend related to the target.
- `is_declining_label` — excluded because it is the target itself.
- `client_id` — excluded because it is an identifier and could allow the model to learn client-specific patterns.
- `content_id` — excluded because it is an identifier rather than a content feature.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
excluded_fields = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "client_id",
    "content_id"
]

print("Excluded fields:")
for field in excluded_fields:
    print("-", field)

Excluded fields:
- trend_direction
- trend_pct
- is_declining_label
- client_id
- content_id


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.